# Obligatorio, Generación de lenguaje natural con Variational Autoencoders

---
- Integrante, número de estudiante
---

# Obligatorio, Generación de lenguaje natural con Variational Autoencoders

Fecha de entrega: 02/12/2026  
Puntaje máximo: 40 puntos  

## Objetivo del obligatorio

Este obligatorio tiene como objetivo principal aplicar un modelo generativo de Deep Learning que no se ha trabajado en clase aún. Se debe diseñar una solución a partir de código boilerplate brindado por el equipo docente, implementación del modelo, evaluación y discusión de la implementación.

## Modelo a implementar

Se debe implementar un **Variational Autoencoder para lenguaje natural**. El corpus de entrenamiento será el dataset **Tiny Shakespeare** de Karpathy, disponible en HuggingFace. La implementación debe basarse en el trabajo [**"Generating Sentences from a Continuous Space"**](https://arxiv.org/abs/1511.06349) de Bowman et al. (2016).

Se deben entrenar **dos variantes** del modelo: una con **RNN vanilla**, y otra a elección usando ya sea **GRUs o LSTMs**.

Además se debe elaborar un **informe académico de máximo 10 carillas** (más anexos) que presente todos los supuestos y decisiones realizados durante el diseño e implementación, evaluación cuantitativa de la solución usando métricas de evaluación de modelos de lenguaje trabajados en el curso, y una evaluación cualitativa general que provenga del uso de la solución.

## Informe académico

- Se debe presentar la implementación elegida además de la que usa RNNs vanilla. También cualquier supuesto realizado en diseño y en experimentación.
- Se debe explorar el hiperparámetro **KL annealing** y mostrar cómo evoluciona la KL a través de su exploración.
- Se deben explorar diferentes **dimensiones de espacio latente**, y mostrar **homotopías** de oraciones del mismo.
- Se debe presentar una evaluación cualitativa de las dos soluciones implementadas *side-to-side* usando métrica de evaluación de modelos de lenguaje estudiados en el curso. Se debe justificar la elección de la métrica e interpretar sus resultados.
- Se debe presentar un **dataset curado de generaciones** con ambas soluciones evaluando puntos fuertes y puntos débiles de los resultados.

## Evaluación

| Objetivo | Puntaje | Descripción |
|---|---:|---|
| **Implementación** | 15 | Se evaluará la implementación correcta del trabajo, que siga un orden coherente y adhiera a buenas prácticas de código y de manejo de pesos, hiperparámetros y artefactos. |
| **Fidelidad al trabajo de base** | 5 | Se evaluará que la implementación sea consistente, salvo supuestos explicados en el informe, con el trabajo propuesto a seguir. |
| **Exploración de diferentes arquitecturas e hiperparámetros** | 5 | Se evaluará la exploración de arquitecturas alternativas y tuning de hiperparámetros. |
| **Evaluación de resultados** | 10 | Se evaluará que el informe y el código fuente contengan los ítems de evaluación de la solución propuestos anteriormente. |
| **Informe** | 5 | Se evaluará la completitud del informe según la sección *Informe académico*. |
| **Total** | **40** | |

## Instrucciones de entrega

- Se entrega este **Jupyter Notebook (.ipynb)** con todo el código ejecutado, más el **informe académico** (máximo 10 carillas, más anexos).
- El notebook debe correr de punta a punta y estar documentado: decisiones de diseño, experimentos realizados y resultados obtenidos.
- Los pesos de los modelos entrenados deben quedar guardados y referenciados, de forma que los resultados del informe sean reproducibles sin volver a entrenar.
- La entrega se realiza a través de gestion.ort.edu.uy antes de la fecha límite.

## Materiales

- [Generating Sentences from a Continuous Space, Bowman et al., 2016](https://arxiv.org/abs/1511.06349)
- [Auto-Encoding Variational Bayes, Kingma & Welling, 2014](https://arxiv.org/abs/1312.6114)
- [Tiny Shakespeare en HuggingFace](https://huggingface.co/datasets/karpathy/tiny_shakespeare)
- [Documentación de PyTorch](https://pytorch.org/docs/stable/index.html)

---

## Requisitos Universitarios

Fecha de entrega: **02/12/2026 hasta las 21:00 horas en gestion.ort.edu.uy** (max. 40Mb en formato zip)

### Uso de material de apoyo y/o consulta

Inteligencia Artificial Generativa:

   - Seguir las pautas de los docentes: Se deben seguir las instrucciones específicas de los docentes sobre cómo utilizar la IA en cada curso.
   - Citar correctamente las fuentes y usos de IA: Siempre que se utilice una herramienta de IA para generar contenido, se debe citar adecuadamente la fuente y la forma en que se utilizó.
   - Verificar el contenido generado por la IA: No todo el contenido generado por la IA es correcto o preciso. Es esencial que los estudiantes verifiquen la información antes de usarla.
   - Ser responsables con el uso de la IA: Conocer los riesgos y desafíos, como la creación de "alucinaciones", los peligros para la privacidad, las cuestiones de propiedad intelectual, los sesgos inherentes y la producción de contenido falso.
   - En caso de existir dudas sobre la autoría, plagio o uso no atribuido de IAG, el docente tendrá la opción de convocar al equipo de obligatorio a una defensa específica e individual sobre el tema.

### Defensa

Fecha de defensa: **07/12/2026, junto al parcial**

La defensa es obligatoria y eliminatoria. El docente es quien definirá y comunicará la modalidad, y mecánica de defensa. La no presentación a la misma implica la pérdida de la totalidad de los puntos del Obligatorio.

IMPORTANTE:

   1) Inscribirse
   2) Formar grupos según lo indicado por el equipo docente
   3) Subir el trabajo a Gestión antes de la hora indicada

Aquellos de ustedes que presenten alguna dificultad con su inscripción o tengan inconvenientes técnicos, por favor contactarse con el Coordinador de cursos o Coordinación adjunta antes de las 20:00h del día de la entrega.

-----------------

## Sobre el modelo

Una VAE cuyo encoder y decoder son redes recurrentes, de modo que el espacio latente continuo ya no representa imágenes sino oraciones.

### El problema que resuelve el paper

Una VAE con decoder recurrente no funciona si se la entrena de forma ingenua. El decoder es de por sí un buen modelo de lenguaje: puede predecir la próxima palabra a partir de las anteriores sin mirar `z`. Al optimizador le conviene entonces llevar el término KL a cero, y la VAE degenera en una RNN común con un espacio latente decorativo. Es el fenómeno que se conoce como posterior collapse.

El paper propone dos mecanismos para evitarlo, y ambos hay que implementarlos:

1. **KL annealing**: el término KL entra multiplicado por un peso que arranca en 0 y sube hasta 1 durante el entrenamiento.

2. **Word dropout**: durante el decoding se reemplazan palabras de la historia por `<unk>`, para debilitar al decoder y forzarlo a apoyarse en `z`.

### Qué está hecho y qué hay que hacer

Vienen resueltas la descarga y tokenización del corpus y el pipeline de datos (vocabulario, dataset y armado de batches). Son procesamiento de lenguaje estándar, no es lo que evalúa este obligatorio. 

El modelo, la pérdida, el entrenamiento, la generación, la evaluación y los experimentos, son lo que se tiene que resolver.

### Imports

In [ ]:
import re
import math
import random
import urllib.request
from collections import Counter

import torch
import torchinfo
import torch.nn as nn
from torch.optim import Adam
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on {DEVICE}")

torch.manual_seed(42)
random.seed(42)

### Dataset: Tiny Shakespeare

El dato ya no es un vector de tamaño fijo sino una secuencia de largo variable de índices enteros sobre un vocabulario. Eso obliga a construir tres cosas: un vocabulario, tokens especiales (`<pad>`, `<unk>`, `<bos>`, `<eos>`) y padding por batch.

#### De dónde sale el corpus

El repositorio [`karpathy/tiny_shakespeare`](https://huggingface.co/datasets/karpathy/tiny_shakespeare) no contiene los datos: contiene únicamente un *loading script*, y ese script descarga el archivo desde

```
https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
```

#### Sobre el tokenizado (esta parte viene resuelta)

Tiny Shakespeare viene en crudo. Partirlo con un `.split()` ingenuo da 25.670 palabras distintas sobre 32.777 líneas, casi una palabra nueva por línea, imposible de aprender, porque `speak.`, `speak,` y `speak` cuentan como tres palabras distintas.

`tiny_shakespeare()` concentra todo el preprocesamiento:

1. Minúsculas y puntuación como token aparte: el vocabulario baja de 25.670 a 11.174.

2. Se descartan las etiquetas de personaje (`ROMEO:`), 8.762 líneas que no son oraciones.

3. Se descartan las líneas de menos de 20 caracteres (1.774) y las de más de 48 (2.934). Las muy cortas no tienen nada que modelar; las largas hacen que el decoder sea tan fuerte que ignore `z`.

4. Se recorta el vocabulario a las 4.000 palabras más frecuentes y se conservan sólo las líneas que quedan 100% cubiertas, lo que descarta otras 7.287. Es el 38% de las que habían sobrevivido, pero a cambio nunca aparece `<unk>` en el texto generado y el token queda libre para que lo use únicamente el word dropout.

Ese 38% no se pierde al azar: se van sistemáticamente las líneas con vocabulario inusual, y las palabras raras son más largas (7,2 caracteres de media contra 6,0 las frecuentes). El corpus resultante no es Shakespeare, es la mitad léxicamente más simple de Shakespeare.

Quedan 12.020 líneas separadas por espacios: de acá en adelante alcanza con `.split()`.

In [ ]:
PAD, UNK, BOS, EOS = 0, 1, 2, 3


def tokenizar(linea):
    """Normalizacion minima: minusculas y puntuacion como token aparte."""
    return re.findall(r"[a-z']+|[.,!?;:]", linea.lower())


def tiny_shakespeare(max_vocab=4000):
    """Descarga Tiny Shakespeare y lo devuelve normalizado y separado por espacios."""
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    texto = urllib.request.urlopen(url).read().decode('utf-8')

    # Una linea = una "oracion". Fuera las etiquetas de personaje y los extremos de largo.
    lineas = [l.strip() for l in texto.split('\n') if l.strip()]
    lineas = [l for l in lineas if not l.endswith(':') and 20 <= len(l) <= 48]

    # Las max_vocab palabras mas frecuentes...
    conteo = Counter(t for l in lineas for t in tokenizar(l))
    frecuentes = {w for w, _ in conteo.most_common(max_vocab)}

    # ...y solo las lineas enteramente cubiertas por ellas.
    return [' '.join(tokenizar(l)) for l in lineas
            if all(t in frecuentes for t in tokenizar(l))]


sentences = tiny_shakespeare()

print(len(sentences), 'oraciones')
for s in sentences[:5]:
    print(' ', s)

### Pipeline de datos

Procesamiento de lenguaje estándar, no es lo que evalúa el obligatorio, por lo que viene resuelto.

`Vocab` traduce entre palabras y enteros, porque los modelos no manipulan texto: `'good'` pasa a ser el número 1.641. Guarda los dos diccionarios, de palabra a índice (`stoi`, string to index) y de índice a palabra (`itos`). Los primeros cuatro índices están reservados:

- `<pad>` (0), relleno para que todas las secuencias de un batch midan lo mismo
- `<unk>` (1), palabra desconocida; acá sólo lo usa el word dropout
- `<bos>` (2), beginning of sentence, marca el arranque
- `<eos>` (3), end of sentence, marca el final

`SentenceDataset` convierte cada oración en un tensor de índices, con `<bos>` adelante y `<eos>` atrás.

`collate` arma el batch. Un `Dataset` devuelve un ejemplo por vez y un `DataLoader` tiene que devolver varios apilados en un solo tensor; esa función de apilado se llama collate. La que trae PyTorch hace `torch.stack` y sirve mientras los ejemplos midan todos lo mismo. Con oraciones de largo distinto falla yhay que escribirla.

Devuelve el batch padeado hasta la oración más larga, el largo real de cada secuencia antes del padding (`lengths`, que el encoder necesita para no resumir la oración con un estado oculto que corresponde a tokens `<pad>`), y las dos versiones desfasadas que usa el decoder:

```
x          <bos>  one   word   ,   good  citizens   .   <eos>
dec_input  <bos>  one   word   ,   good  citizens   .            - lo que VE el decoder
target            one   word   ,   good  citizens   .   <eos>    - lo que debe predecir
```

El decoder es autorregresivo: en cada posición recibe un token y predice el siguiente, de ahí el corrimiento. `<bos>` está en `dec_input` pero no en `target`, porque nunca hay que generarlo; `<eos>` está en `target` pero no en `dec_input`, porque es lo que le enseña al modelo cuándo parar.

Por último se separa un conjunto de evaluación, y el vocabulario se construye sólo con las oraciones de entrenamiento. Si se armara con el corpus entero, el modelo entraría sabiendo algo de las oraciones con las que después se lo mide.

In [ ]:
class Vocab:
    """Traduce entre palabras e indices enteros."""

    def __init__(self, sentences, min_freq=1, tokenize=str.split, sep=' '):
        # tokenize=str.split -> nivel palabra;  tokenize=list -> nivel caracter
        self.tokenize, self.sep = tokenize, sep
        counter = Counter(w for s in sentences for w in tokenize(s))
        palabras = [w for w, c in counter.items() if c >= min_freq]
        # los cuatro primeros indices son los especiales: coinciden con las
        # constantes PAD, UNK, BOS, EOS definidas arriba
        self.itos = ['<pad>', '<unk>', '<bos>', '<eos>'] + sorted(palabras)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, s):
        """'good citizens' -> [1641, 905]"""
        return [self.stoi.get(w, UNK) for w in self.tokenize(s)]

    def decode(self, ids):
        """[1641, 905] -> 'good citizens'. Omite los tokens especiales."""
        return self.sep.join(self.itos[i] for i in ids if i not in (PAD, BOS, EOS))


class SentenceDataset(Dataset):
    """Cada oracion pasa a ser un tensor de indices, con <bos> y <eos>."""

    def __init__(self, sentences, vocab, max_len=30):
        # el truncado va ANTES de agregar <eos>, para que el corte no se lo coma
        self.data = [[BOS] + vocab.encode(s)[:max_len] + [EOS] for s in sentences]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return torch.tensor(self.data[i])


def collate(batch):
    """Apila una lista de secuencias de largo variable en un unico batch.

    Devuelve (x, lengths, dec_input, target):
      x         (B, T)   secuencias padeadas con PAD
      lengths   (B,)     largo real de cada secuencia, antes del padding
      dec_input (B, T-1) lo que recibe el decoder en cada paso
      target    (B, T-1) lo que el decoder debe predecir en cada paso
    """
    lengths = torch.tensor([len(s) for s in batch])
    x = nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=PAD)
    # las dos vistas, desfasadas una posicion: el decoder ve x[:-1] y predice x[1:]
    return x, lengths, x[:, :-1], x[:, 1:]


# El split va ANTES de construir el vocabulario, para no filtrar informacion
# del conjunto de evaluacion dentro del modelo.
random.shuffle(sentences)
n_eval = int(0.1 * len(sentences))
eval_sentences, train_sentences = sentences[:n_eval], sentences[n_eval:]

vocab = Vocab(train_sentences)
batch_size = 64
train_loader = DataLoader(SentenceDataset(train_sentences, vocab), batch_size=batch_size,
                          shuffle=True, collate_fn=collate)
eval_loader = DataLoader(SentenceDataset(eval_sentences, vocab), batch_size=batch_size,
                         shuffle=False, collate_fn=collate)

print(f'train {len(train_sentences)} oraciones | eval {len(eval_sentences)} oraciones')
print('vocabulario:', len(vocab), 'tokens')

x, lengths, dec_input, target = next(iter(train_loader))
print()
print('x        ', tuple(x.shape), '<- (batch, largo maximo del batch)')
print('lengths  ', tuple(lengths.shape))
print('dec_input', tuple(dec_input.shape))
print('target   ', tuple(target.shape))
print()
ver = lambda t: ' '.join(vocab.itos[i] for i in t.tolist())
print('x        ', ver(x[0]))
print('dec_input', ver(dec_input[0]))
print('target   ', ver(target[0]))

### Modelo: SentenceVAE

La arquitectura está en la sección 3 del paper. Las firmas de abajo fijan el contrato con el resto del notebook; todo lo demás es decisión de ustedes y va justificado en el informe.

In [ ]:
class SentenceVAE(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=256, latent_dim=16,
                 rnn_type='lstm', word_dropout=0.5, device='cpu'):
        """rnn_type es 'rnn' (vanilla), 'gru' o 'lstm'."""
        super(SentenceVAE, self).__init__()
        # TODO
        ...

    def encode(self, x, lengths):
        """(B, T) + (B,) -> mean (B, latent_dim), log_var (B, latent_dim)."""
        # TODO
        ...

    def reparameterization(self, mean, log_var):
        # TODO
        ...

    def decode(self, z, dec_input):
        """z (B, latent_dim) + dec_input (B, T) -> logits (B, T, vocab_size)."""
        # TODO
        ...

    def forward(self, x, lengths, dec_input):
        """-> logits (B, T, vocab_size), mean (B, latent_dim), log_var (B, latent_dim)."""
        # TODO
        ...

    def generate(self, num_samples=1, max_len=20, temperature=1.0, z=None):
        """Si z es None muestrea del prior, y si se pasa un z decodifica ese punto.

        Devuelve un tensor (num_samples, L) de indices.
        """
        # TODO
        ...

In [ ]:
LR = ...

model = SentenceVAE(vocab_size=len(vocab), rnn_type='lstm', device=DEVICE).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)

torchinfo.summary(model, input_data=[x.to(DEVICE), lengths, dec_input.to(DEVICE)],
                  col_names=('input_size', 'output_size', 'num_params'))

###  Función de Pérdida

In [ ]:
def loss_function(logits, target, mean, log_var, kl_weight=1.0):
    """-> (loss_total, reconstruccion, KL sin ponderar)"""
    # TODO
    ...


def kl_weight_at(step, total_anneal_steps):
    """Peso del termino KL en el paso 'step' (seccion 3.1 del paper).

    Con total_anneal_steps <= 0 debe devolver 1.0: es el caso sin annealing.
    """
    # TODO
    ...

### Entrenamiento

In [ ]:
def train(model, optimizer, epochs, device, anneal_epochs=6, log_every=1):
    """Entrena y devuelve la historia: una lista con un (recon, KL) por epoca,
    ambos promediados por oracion."""
    # TODO
    ...

In [ ]:
historia = train(model, optimizer, epochs=15, device=DEVICE, log_every=3)

In [ ]:
# Diagnostico (dado): la curva de la derecha es la que decide si el modelo sirve.
rec, kl = zip(*historia)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(rec); ax[0].set_title('Reconstrucción por oración'); ax[0].set_xlabel('época')
ax[1].plot(kl);  ax[1].set_title('KL por oración');             ax[1].set_xlabel('época')
ax[1].axhline(0, color='red', ls='--', lw=1, label='colapso')
ax[1].legend()
plt.tight_layout(); plt.show()

print('KL final:', round(kl[-1], 3), 'nats por oración')
print('  KL ~ 0  -> colapso: el decoder ignora z y esto ya no es una VAE')
print('  KL > 1  -> z está transportando información real sobre la oración')

### Generación

In [ ]:
def generate_sentences(model, n=10, temperature=1.0):
    tokens = model.generate(num_samples=n, temperature=temperature)
    for fila in tokens.cpu().tolist():
        print(' ', vocab.decode(fila))


generate_sentences(model, n=10)

### Evaluación cuantitativa

In [ ]:
def evaluar(model, loader):
    """Evalua el modelo sobre el conjunto de evaluacion."""
    # TODO
    ...


evaluar(model, eval_loader)

### El espacio latente: homotopías

Es la figura del paper (Tabla 8) y el entregable visual del obligatorio. Se toman dos oraciones reales, se las codifica a $z_1$ y $z_2$, y se decodifican los puntos intermedios de la recta que los une:

$$z_t = (1-t)\,z_1 + t\,z_2, \qquad t \in [0, 1]$$

Si el espacio latente es continuo y está bien organizado, **todos los puntos intermedios son oraciones gramaticales** y la transición de una a otra es gradual. Si el modelo colapsó, pasa una de dos: no cambia nada a lo largo de la recta, o se salta de una oración a la otra con basura en el medio.

In [ ]:
def homotopia(model, frase_a, frase_b, pasos=8):
    # TODO
    ...


homotopia(model, train_sentences[0], train_sentences[-1])